# 🧊 TRELLIS — Image to 3D Model (Colab Edition)

> **Microsoft TRELLIS** — แปลงรูปภาพเป็น 3D Model (.glb / .obj) ด้วย AI  
> Paper: [Structured 3D Latents for Scalable and Versatile 3D Generation](https://arxiv.org/abs/2412.01506) (CVPR 2025)

---

## ✅ วิธีใช้งาน
1. **Runtime → Change runtime type → GPU** (T4 ขึ้นไป)
2. กด **Runtime → Run all** รอการติดตั้ง (~20-30 นาที)
3. เมื่อมี Gradio Link ขึ้นมา → คลิกลิงก์ → อัปโหลดรูป → กด Generate!

---
| Cell | หัวข้อ | รายละเอียด |
|------|--------|------------|
| 1 | ตรวจสอบ GPU | nvidia-smi |
| 2 | ติดตั้ง CUDA 11.8 | NVIDIA repo |
| 3 | ติดตั้ง condacolab | Miniconda บน Colab |
| 4 | สร้าง Conda env | Python 3.11 + PyTorch 2.3 |
| 5 | ติดตั้ง System tools | cmake, ninja, etc. |
| 6 | Clone TRELLIS | GitHub + submodules |
| 7 | Build TRELLIS | setup.sh |
| 8 | Clone mip-splatting | Gaussian splatting |
| 9 | รัน Gradio App | เปิด UI |


## 🔍 Cell 1 — ตรวจสอบ GPU และ Environment

In [ ]:
import subprocess, sys

# ตรวจสอบ GPU
print('=' * 60)
print('🖥️  GPU INFO')
print('=' * 60)
!nvidia-smi

# ตรวจสอบ RAM
print('\n' + '=' * 60)
print('💾  SYSTEM RAM')
print('=' * 60)
!free -h

# ตรวจสอบ Disk
print('\n' + '=' * 60)
print('📀  DISK SPACE')
print('=' * 60)
!df -h /content

# ตรวจสอบ CUDA ที่มีอยู่
print('\n' + '=' * 60)
print('⚡  CUDA VERSION')
print('=' * 60)
!nvcc --version 2>/dev/null || echo '(nvcc not yet installed — will install CUDA 11.8 in next cell)'
print('\n✅ GPU check done!')

## ⚡ Cell 2 — ติดตั้ง CUDA 11.8
> TRELLIS ต้องการ CUDA 11.8 ที่ pin ไว้สำหรับ compatibility กับ flash-attn, spconv, kaolin

In [ ]:
%%bash
set -e
echo '========================================'
echo '⚡ Installing CUDA 11.8...'
echo '========================================'

# อัปเดต apt
apt-get update -y -qq
apt-get install -y -qq gnupg wget curl

# เพิ่ม NVIDIA CUDA keyring
wget -q https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/cuda-keyring_1.1-1_all.deb
dpkg -i cuda-keyring_1.1-1_all.deb
apt-get update -y -qq

# ติดตั้ง CUDA 11.8
apt-get install -y -qq cuda-11-8

# ตั้งค่า PATH
echo 'export PATH="/usr/local/cuda-11.8/bin:$PATH"' >> ~/.bashrc
echo 'export LD_LIBRARY_PATH="/usr/local/cuda-11.8/lib64:$LD_LIBRARY_PATH"' >> ~/.bashrc
export PATH="/usr/local/cuda-11.8/bin:$PATH"
export LD_LIBRARY_PATH="/usr/local/cuda-11.8/lib64:$LD_LIBRARY_PATH"

# ตรวจสอบ
nvcc --version
echo ''
echo '✅ CUDA 11.8 installed successfully!'

## 🐍 Cell 3 — ติดตั้ง condacolab และ Miniconda
> ⚠️ **Kernel จะ restart อัตโนมัติ** หลัง cell นี้เสร็จ — นั่นคือปกติ! รอแล้วรัน Cell 4 ต่อ

In [ ]:
import sys
print('=' * 60)
print('🐍 Installing condacolab + Miniconda...')
print('=' * 60)

!pip install -q condacolab

import condacolab
condacolab.install_miniconda()
# ⚠️ Kernel จะ restart ตรงนี้ — เป็นเรื่องปกติ

## 📦 Cell 4 — สร้าง Conda Environment + ติดตั้ง PyTorch และ Dependencies
> Pin versions: PyTorch 2.3.0 + CUDA 11.8 + flash-attn 2.5.8 + spconv + kaolin 0.17.0

In [ ]:
%%bash
set -e
source /usr/local/etc/profile.d/conda.sh

echo '========================================'
echo '📦 Creating conda environment: trellis'
echo '     Python 3.11 | PyTorch 2.3.0 | CUDA 11.8'
echo '========================================'

# ลบ env เก่า (ถ้ามี)
conda remove -n trellis --all -y 2>/dev/null || true

# สร้าง env ใหม่
conda create -y -n trellis python=3.11
conda activate trellis

echo ''
echo '📥 Installing PyTorch 2.3.0 + CUDA 11.8...'
conda install -y \
    pytorch==2.3.0 \
    torchvision==0.18.0 \
    torchaudio==2.3.0 \
    pytorch-cuda=11.8 \
    -c pytorch -c nvidia

echo ''
echo '📥 Installing FlashAttention 2.5.8...'
pip install -q \
    https://github.com/Dao-AILab/flash-attention/releases/download/v2.5.8/flash_attn-2.5.8+cu118torch2.3cxx11abiFALSE-cp311-cp311-linux_x86_64.whl

echo ''
echo '📥 Installing spconv-cu118...'
pip install -q spconv-cu118

echo ''
echo '📥 Installing Kaolin 0.17.0...'
pip install -q kaolin==0.17.0 \
    -f https://nvidia-kaolin.s3.us-east-2.amazonaws.com/torch-2.3.0_cu118.html

echo ''
echo '📥 Installing Pillow (pinned < 11.0), Gradio...'
pip install -q --force-reinstall 'pillow<11.0'
pip install -q 'gradio==4.44.1' 'gradio_litmodel3d==0.0.1'

echo ''
echo '📥 Installing other essential packages...'
pip install -q \
    easydict \
    einops \
    huggingface_hub \
    imageio \
    imageio-ffmpeg \
    largestinteriorrectangle \
    matplotlib \
    numpy \
    omegaconf \
    open3d \
    opencv-python-headless \
    rembg \
    safetensors \
    scikit-image \
    scipy \
    timm \
    tqdm \
    trimesh \
    transformers \
    xatlas

echo ''
echo '✅ Conda environment created and all packages installed!'

## 🔧 Cell 5 — ติดตั้ง System Build Tools
> cmake, ninja-build, nvidia-cuda-toolkit สำหรับ compile C++ extensions

In [ ]:
%%bash
set -e

echo '========================================'
echo '🔧 Installing system build tools...'
echo '========================================'

apt-get update -y -qq
apt-get install -y -qq \
    build-essential \
    cmake \
    ninja-build \
    nvidia-cuda-toolkit \
    libgl1-mesa-dev \
    libglib2.0-0 \
    libsm6 \
    libxext6 \
    libxrender-dev \
    libgomp1

# ตั้งค่า CUDA PATH
export PATH="/usr/local/cuda-11.8/bin:$PATH"
export LD_LIBRARY_PATH="/usr/local/cuda-11.8/lib64:$LD_LIBRARY_PATH"
export CUDA_HOME="/usr/local/cuda-11.8"

# อัปเกรด Python build tools ใน conda env
source /usr/local/etc/profile.d/conda.sh
conda activate trellis
pip install -q --upgrade pip setuptools wheel ninja Cython

echo ''
echo '✅ Build tools installed!'
gcc --version | head -1
cmake --version | head -1
ninja --version

## 📂 Cell 6 — Clone TRELLIS Repository + Submodules

In [ ]:
%%bash
set -e

echo '========================================'
echo '📂 Cloning Microsoft TRELLIS...'
echo '========================================'

# ลบโฟลเดอร์เก่า
rm -rf /content/TRELLIS

# Clone พร้อม submodules
git clone --recurse-submodules \
    https://github.com/microsoft/TRELLIS.git \
    /content/TRELLIS

cd /content/TRELLIS

# อัปเดต submodules
git submodule update --init --recursive

# สร้างโฟลเดอร์ที่จำเป็น (ป้องกัน app.py crash)
mkdir -p /content/TRELLIS/assets/example_image
mkdir -p /content/TRELLIS/outputs

echo ''
echo 'Repository structure:'
ls -la /content/TRELLIS/

echo ''
echo '✅ TRELLIS cloned successfully!'

## 🏗️ Cell 7 — Build TRELLIS (setup.sh)
> ขั้นตอนนี้ใช้เวลานาน ~15-25 นาที — compile C++ extensions (nvdiffrast, mipgaussian, kaolin ฯลฯ)

In [ ]:
%%bash
set -e

echo '========================================'
echo '🏗️  Building TRELLIS extensions...'
echo '    (อาจใช้เวลา 15-25 นาที)'
echo '========================================'

export PATH="/usr/local/cuda-11.8/bin:$PATH"
export LD_LIBRARY_PATH="/usr/local/cuda-11.8/lib64:$LD_LIBRARY_PATH"
export CUDA_HOME="/usr/local/cuda-11.8"
export TORCH_CUDA_ARCH_LIST="7.0;7.5;8.0;8.6+PTX"  # ครอบคลุม T4/A100/V100

source /usr/local/etc/profile.d/conda.sh
conda activate trellis

cd /content/TRELLIS

# รัน setup.sh
bash ./setup.sh \
    --basic \
    --flash-attn \
    --spconv \
    --mipgaussian \
    --nvdiffrast \
    --kaolin \
    --demo

echo ''
echo '✅ TRELLIS build complete!'

## 🌟 Cell 8 — Clone mip-splatting (Gaussian Rasterization)
> ใช้สำหรับ Gaussian Splatting output format

In [ ]:
%%bash
set -e

echo '========================================'
echo '🌟 Cloning mip-splatting...'
echo '========================================'

export PATH="/usr/local/cuda-11.8/bin:$PATH"
export LD_LIBRARY_PATH="/usr/local/cuda-11.8/lib64:$LD_LIBRARY_PATH"
export CUDA_HOME="/usr/local/cuda-11.8"
export TORCH_CUDA_ARCH_LIST="7.0;7.5;8.0;8.6+PTX"

source /usr/local/etc/profile.d/conda.sh
conda activate trellis

cd /content

# ลบโฟลเดอร์เก่า
rm -rf /content/mip-splatting

# Clone
git clone --recurse-submodules \
    https://github.com/autonomousvision/mip-splatting.git

cd /content/mip-splatting
git submodule update --init --recursive

# ติดตั้ง OpenGL dependencies
apt-get install -y -qq libgl1-mesa-dev

# Build diff_gaussian_rasterization
cd /content/mip-splatting/submodules/diff-gaussian-rasterization
pip install . --no-build-isolation -v

echo ''
echo '✅ mip-splatting installed!'

## 🎨 Cell 9 — Patch app.py + ดาวน์โหลด Model Weights
> Patch ให้ Gradio รัน share=True และ download weights จาก HuggingFace

In [ ]:
import os

APP_PY = '/content/TRELLIS/app.py'

print('=' * 60)
print('🔧 Patching app.py...')
print('=' * 60)

with open(APP_PY, 'r') as f:
    content = f.read()

# Patch: ให้ launch ด้วย share=True และ port ที่กำหนด
patches = [
    ('demo.launch()', 'demo.launch(share=True, server_port=7860)'),
    ('demo.launch(share=False)', 'demo.launch(share=True, server_port=7860)'),
]

for old, new in patches:
    if old in content:
        content = content.replace(old, new)
        print(f'  ✅ Patched: {old!r} → {new!r}')

with open(APP_PY, 'w') as f:
    f.write(content)

print('\n📋 app.py launch line:')
for line in content.split('\n'):
    if 'launch' in line:
        print('  ', line.strip())

print('\n✅ Patch done!')

## 🚀 Cell 10 — รัน TRELLIS Gradio App
> หลังรัน cell นี้ จะมี **Public URL (share link)** ปรากฏ — คลิกเพื่อเปิด UI
>
> 🕐 รอประมาณ 2-5 นาที สำหรับ download model weights ครั้งแรก

In [ ]:
import subprocess
import os

print('=' * 60)
print('🚀 Starting TRELLIS Gradio App...')
print('=' * 60)
print('📌 รอ URL ที่ขึ้นว่า: Running on public URL: https://xxxx.gradio.live')
print('   (อาจใช้เวลา 2-5 นาทีแรก สำหรับ download model weights)')
print()

# ตั้งค่า environment
env = os.environ.copy()
env.update({
    'PATH': '/usr/local/cuda-11.8/bin:' + env.get('PATH', ''),
    'LD_LIBRARY_PATH': '/usr/local/cuda-11.8/lib64:' + env.get('LD_LIBRARY_PATH', ''),
    'CUDA_HOME': '/usr/local/cuda-11.8',
    'OPENCV_IO_ENABLE_OPENEXR': '1',
    'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',  # ประหยัด VRAM
})

# รัน app.py ใน conda env trellis
proc = subprocess.Popen(
    ['conda', 'run', '-n', 'trellis', 'python', '-u', 'app.py'],
    cwd='/content/TRELLIS',
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    env=env
)

try:
    for line in iter(proc.stdout.readline, b''):
        decoded = line.decode('utf-8', errors='replace').rstrip()
        print(decoded)
        # Highlight URL
        if 'gradio.live' in decoded or 'Running on' in decoded:
            print('\n' + '🎉 ' * 10)
            print('🔗 คลิก URL ด้านบนเพื่อเปิด TRELLIS!')
            print('🎉 ' * 10 + '\n')
except KeyboardInterrupt:
    print('\n⛔ หยุดการทำงาน')
    proc.kill()
finally:
    proc.wait()

---
## 🔧 Optional Cells — ใช้ได้ตามต้องการ

### 🌐 Optional A — รันด้วย Ngrok (ถ้า Gradio share ไม่ทำงาน)
> ต้องสมัคร [ngrok.com](https://ngrok.com) และเอา Auth Token มาใส่

In [ ]:
# ============================================================
# ใส่ ngrok auth token ของคุณที่นี่
NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTH_TOKEN"  # ← แก้ตรงนี้!
# ============================================================

import subprocess, time, json, os

if NGROK_AUTH_TOKEN == "YOUR_NGROK_AUTH_TOKEN":
    print("⚠️  กรุณาใส่ NGROK_AUTH_TOKEN ก่อน!")
    print("   สมัครฟรีที่: https://ngrok.com")
else:
    print('=' * 60)
    print('🌐 Starting Ngrok tunnel...')
    print('=' * 60)

    # ติดตั้ง ngrok
    !pip install -q pyngrok
    from pyngrok import ngrok, conf

    # ตั้งค่า auth token
    conf.get_default().auth_token = NGROK_AUTH_TOKEN
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

    # เปิด tunnel ที่ port 7860
    tunnel = ngrok.connect(7860)
    print(f'\n🔗 NGROK Public URL: {tunnel.public_url}')
    print('📌 เปิด URL ด้านบนในเบราว์เซอร์!')

    # รัน app (non-patched version ก็ได้ เพราะ ngrok จัดการ tunnel แล้ว)
    env = os.environ.copy()
    env.update({
        'PATH': '/usr/local/cuda-11.8/bin:' + env.get('PATH', ''),
        'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
    })

    proc = subprocess.Popen(
        ['conda', 'run', '-n', 'trellis', 'python', '-u', 'app.py'],
        cwd='/content/TRELLIS',
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=env
    )

    try:
        for line in iter(proc.stdout.readline, b''):
            print(line.decode('utf-8', errors='replace').rstrip())
    except KeyboardInterrupt:
        proc.kill()
    finally:
        ngrok.disconnect(tunnel.public_url)
        proc.wait()

### 🐍 Optional B — รัน Inference โดยตรงด้วย Python (ไม่ใช้ UI)
> อัปโหลดรูปภาพไว้ที่ `/content/my_image.png` แล้วรัน cell นี้

In [ ]:
# ============================================================
# ตั้งค่าที่นี่
IMAGE_PATH = "/content/my_image.png"   # ← path ของรูปภาพ
OUTPUT_DIR = "/content/outputs"        # ← โฟลเดอร์สำหรับ output
SEED       = 42                         # ← seed สำหรับ reproducibility
STEPS      = 12                         # ← จำนวน steps (12-25)
# ============================================================

import os, sys
sys.path.insert(0, '/content/TRELLIS')
os.makedirs(OUTPUT_DIR, exist_ok=True)

os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import torch
from PIL import Image
from trellis.pipelines import TrellisImageTo3DPipeline
from trellis.utils import render_utils, postprocessing_utils

print('=' * 60)
print('🎨 TRELLIS Image-to-3D Inference')
print('=' * 60)

# โหลด Pipeline
print('📥 Loading TRELLIS-image-large model...')
pipeline = TrellisImageTo3DPipeline.from_pretrained(
    "JeffreyXiang/TRELLIS-image-large"
)
pipeline = pipeline.cuda()
print('✅ Model loaded!')

# โหลดรูปภาพ
print(f'\n🖼️  Loading image: {IMAGE_PATH}')
if not os.path.exists(IMAGE_PATH):
    raise FileNotFoundError(f"ไม่พบรูปภาพ: {IMAGE_PATH}")
image = Image.open(IMAGE_PATH).convert('RGBA')
print(f'   Image size: {image.size}')

# Generate 3D
print(f'\n🚀 Generating 3D model (seed={SEED}, steps={STEPS})...')
outputs = pipeline.run(
    image,
    seed=SEED,
    formats=["mesh", "gaussian"],
    preprocess_image=True,
    sparse_structure_sampler_params={
        "steps": STEPS,
        "cfg_strength": 7.5,
    },
    slat_sampler_params={
        "steps": STEPS,
        "cfg_strength": 3,
    },
)
print('✅ Generation complete!')

# Export GLB
glb_path = os.path.join(OUTPUT_DIR, 'output.glb')
print(f'\n💾 Exporting GLB to: {glb_path}')
glb = postprocessing_utils.to_glb(
    outputs['gaussian'][0],
    outputs['mesh'][0],
    simplify=0.95,
    texture_size=1024,
)
glb.export(glb_path)

# แสดงผล
print('\n' + '=' * 60)
print('✅ 3D Model saved!')
print(f'   📁 {glb_path}')
print('\n💡 วิธีดาวน์โหลด:')
print('   Files panel (ซ้ายมือ) → /content/outputs → คลิกขวา → Download')
print('=' * 60)

# Download อัตโนมัติ
try:
    from google.colab import files
    files.download(glb_path)
except Exception:
    print('(ดาวน์โหลด manual จาก Files panel)')

---
## 🛠️ Troubleshooting

| ปัญหา | วิธีแก้ |
|-------|--------|
| `CUDA out of memory` | Runtime → Factory Reset Runtime แล้วรัน Run All ใหม่ |
| Kernel crash หลัง condacolab | ปกติ! รัน Cell 4 ต่อได้เลย |
| Gradio link ไม่ขึ้น | ใช้ Optional A (Ngrok) แทน |
| `ModuleNotFoundError` | ตรวจสอบว่า conda activate trellis ก่อน import |
| รูปไม่มี background ใส | ลองรูปที่มี background ขาว/เรียบๆ หรือ png ที่มี alpha channel |

---
## 📚 References
- [Microsoft TRELLIS GitHub](https://github.com/microsoft/TRELLIS)
- [TRELLIS-image-large (HuggingFace)](https://huggingface.co/JeffreyXiang/TRELLIS-image-large)
- [Paper: arXiv 2412.01506](https://arxiv.org/abs/2412.01506)
- [mip-splatting](https://github.com/autonomousvision/mip-splatting)
- [FlashAttention](https://github.com/Dao-AILab/flash-attention)
- [Kaolin (NVIDIA)](https://github.com/NVIDIAGameWorks/kaolin)
